A retail company maintains a product catalog in its data warehouse.
Product details such as name, category, and price may change over time due to rebranding, category updates, or pricing adjustments. To preserve historical data for accurate reporting and trend analysis, the company needs to implement a Slowly Changing Dimension (SCD) Type 2 mechanism in PySpark, ensuring old records are retained with effective date ranges while new versions are inserted as separate records.

In [0]:
%sql
CREATE TABLE pyspark_cata.source.customers
(
  id STRING,
  email STRING,
  city STRING,
  country STRING,
  modifiedDate TIMESTAMP
)

In [0]:
%sql
INSERT INTO pyspark_cata.source.customers
VALUES
('1', 'john.smith@example.com', 'New York', 'USA', current_timestamp),
('2', 'jane.doe@example.com', 'London', 'UK',current_timestamp),
('3', 'bob.smith@example.com', 'Paris', 'France',current_timestamp),
('4', 'alice.jones@example.com', 'Tokyo', 'Japan',current_timestamp),
('5', 'charlie.williams@example.com', 'Sydney', 'Australia',current_timestamp)

In [0]:
%sql
INSERT INTO pyspark_cata.source.customers
VALUES
('3', 'bob.smith@example.com', 'Lyon', 'France',current_timestamp),
('7', 'jane.doe@example.com',  'Tokyo', 'Japan',current_timestamp)

In [0]:
%sql
DELETE FROM pyspark_cata.source.customers
WHERE city = 'New York'

In [0]:
%sql
SELECT * FROM pyspark_cata.source.customers

In [0]:
if spark.catalog.tableExists("pyspark_cata.source.DimCustomers"):
    pass
else:
    spark.sql(
        """
        CREATE TABLE pyspark_cata.source.DimCustomers
        SELECT *,
        current_timestamp() as startTime,
        CAST('3000-01-01' AS timestamp) as endTime,
        'Y' as isActive
        FROM pyspark_cata.source.customers
        """
    )

In [0]:
%sql
SELECT * FROM pyspark_cata.source.DimCustomers

### **SCD- TYPE 2**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
df = spark.sql(
    """
    SELECT * FROM pyspark_cata.source.customers
    """
)

df= df.withColumn("dedup", row_number().over(Window.partitionBy("id").orderBy(col("modifiedDate").desc())))\
    .drop("dedup")

df = df.filter(col("dedup") == 1)

df.createOrReplaceTempView('srctemp')

df = spark.sql(
        """
            SELECT *,
            current_timestamp() as startTime,
            CAST('3000-01-01' AS timestamp) as endTime,
            'Y' as isActive
            FROM srctemp
        """
    )

df.createOrReplaceTempView('src')

In [0]:
%sql
SELECT * FROM src

### **MERGE 1 - Marking the updated records as expired**

In [0]:
%sql
MERGE INTO pyspark_cata.source.DimCustomers as tgt
USING src as src
ON tgt.id = src.id
AND tgt.isActive = 'Y'

WHEN MATCHED AND src.email != tgt.email 
OR src.city != tgt.city
OR src.country != tgt.country
OR src.modifiedDate != tgt.modifiedDate

THEN UPDATE SET 
tgt.endTime = current_timestamp(),
tgt.isActive = 'N'

### **MERGE 2 - INSERTING NEW + UPDTAED RECORDS**

In [0]:
%sql
MERGE INTO pyspark_cata.source.DimCustomers as tgt
USING src as src
ON src.id = tgt.id
AND tgt.isActive = 'Y'

WHEN NOT MATCHED THEN INSERT *


In [0]:
%sql
SELECT * FROM pyspark_cata.source.DimCustomers